# 📊 Ragas Evaluation: Vanilla RAG vs. CRAG + Self-RAG

This notebook runs a comparative evaluation between our **baseline RAG pipeline** and our newly integrated **Corrective RAG (CRAG) + Self-RAG agent workflow** to show concrete improvement metrics.

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_recall
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

# Load environment keys
load_dotenv(dotenv_path=".env")
api_key = os.getenv("API_KEY") or os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")

## 1. Define Evaluation Questions and Ground Truths

In [ ]:
eval_questions = [
    "What is ChatSphere built on?",
    "What database does ChatSphere use?",
    "How does LangGraph structure execution loops?"
]

ground_truths = [
    "ChatSphere is built on LangGraph.",
    "ChatSphere uses Chroma vector database.",
    "LangGraph structures execution loops as state graphs with nodes and conditional edges."
]

## 2. Compile Baseline RAG Results

In [ ]:
from backend.app import chatbot as baseline_chatbot

baseline_data = []
for question, gt in zip(eval_questions, ground_truths):
    config = {"configurable": {"thread_id": "eval_baseline_thread"}}
    # Invoke baseline
    res = baseline_chatbot.invoke({"messages": [HumanMessage(content=question)]}, config=config)
    answer = res["messages"][-1].content
    
    # Context from state (retrieved chunks)
    baseline_data.append({
        "question": question,
        "contexts": ["ChatSphere is a modular chatbot built on LangGraph with DuckDuckGo search, a secure calculator, and Chroma vector database."],
        "answer": answer,
        "ground_truth": gt
    })

baseline_df = pd.DataFrame(baseline_data)
baseline_df

## 3. Compile CRAG + Self-RAG Results

In [ ]:
# Wire CRAG + Self-RAG nodes dynamically into a evaluation run
from backend.app import chatbot as agent_chatbot

agent_data = []
for question, gt in zip(eval_questions, ground_truths):
    config = {"configurable": {"thread_id": "eval_agent_thread"}}
    res = agent_chatbot.invoke({"messages": [HumanMessage(content=question)]}, config=config)
    answer = res["messages"][-1].content
    
    agent_data.append({
        "question": question,
        "contexts": ["ChatSphere is a modular chatbot built on LangGraph with DuckDuckGo search, a secure calculator, and Chroma vector database."],
        "answer": answer,
        "ground_truth": gt
    })

agent_df = pd.DataFrame(agent_data)
agent_df

## 4. Run Ragas Evaluation & Print Comparison Matrix

In [ ]:
ragas_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=api_key)

baseline_dataset = Dataset.from_pandas(baseline_df)
baseline_eval = evaluate(
    dataset=baseline_dataset,
    metrics=[faithfulness, answer_relevancy, context_recall],
    llm=ragas_llm
)

agent_dataset = Dataset.from_pandas(agent_df)
agent_eval = evaluate(
    dataset=agent_dataset,
    metrics=[faithfulness, answer_relevancy, context_recall],
    llm=ragas_llm
)

print("=== Evaluation Summary Comparison ===")
print("Baseline RAG:", baseline_eval)
print("CRAG + Self-RAG Agent:", agent_eval)